# Práctica 1 — Q-Learning Tabular
**Materia:** Aprendizaje por Refuerzo  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Implementar el algoritmo Q-Learning tabular sobre el entorno FrozenLake de Gymnasium y analizar la influencia de los hiperparámetros α (tasa de aprendizaje) y γ (factor de descuento).

In [ ]:
!pip install gymnasium --quiet

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random
np.random.seed(42); random.seed(42)

## 1. Entorno FrozenLake

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=False)
n_states  = env.observation_space.n
n_actions = env.action_space.n
print(f"Estados: {n_states} | Acciones: {n_actions}")
print("Mapa del entorno:")
env.reset(); env.render()

## 2. Inicialización de la Q-table y parámetros

In [ ]:
Q = np.zeros((n_states, n_actions))
ALPHA   = 0.8    # tasa de aprendizaje
GAMMA   = 0.95   # factor de descuento
EPSILON = 1.0    # exploración inicial
EPS_MIN = 0.01
EPS_DEC = 0.995
EPISODIOS = 2000
recompensas = []

## 3. Bucle de entrenamiento

In [ ]:
for ep in range(EPISODIOS):
    estado, _ = env.reset()
    total_r = 0
    for _ in range(100):
        if random.random() < EPSILON:
            accion = env.action_space.sample()
        else:
            accion = np.argmax(Q[estado])

        nuevo_estado, recompensa, terminado, truncado, _ = env.step(accion)
        Q[estado, accion] += ALPHA * (
            recompensa + GAMMA * np.max(Q[nuevo_estado]) - Q[estado, accion]
        )
        estado = nuevo_estado
        total_r += recompensa
        if terminado or truncado:
            break

    EPSILON = max(EPS_MIN, EPSILON * EPS_DEC)
    recompensas.append(total_r)

print(f"Recompensa promedio (últimos 100 ep): {np.mean(recompensas[-100:]):.3f}")

## 4. Resultados

In [ ]:
ventana = 50
medias = [np.mean(recompensas[max(0,i-ventana):i+1]) for i in range(len(recompensas))]
plt.figure(figsize=(10,4))
plt.plot(recompensas, alpha=0.3, color='steelblue', label='Recompensa ep.')
plt.plot(medias, color='navy', linewidth=2, label=f'Media móvil ({ventana} ep.)')
plt.xlabel('Episodio'); plt.ylabel('Recompensa')
plt.title('Aprendizaje Q-Learning — FrozenLake')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('qlearning.png', dpi=120); plt.show()

In [ ]:
print("Q-table aprendida:")
print(np.round(Q, 3))

## Conclusiones
- Q-Learning converge a la política óptima en entornos deterministas con suficientes episodios.
- La exploración ε-greedy con decaimiento exponencial equilibra exploración y explotación.
- Para entornos con espacio de estados continuo, Q-tabular no escala; se requiere Deep Q-Network (DQN).